In [ ]:
ItemId = "" # Optional single-model override; in this starter only the OneLake group triggers a refresh
WorkspaceName = "HelixFabric-Insights"

# OneLake security errors affect all models sharing the same lakehouse.
# When ItemId is empty or the OneLake group sentinel, refresh all models below.
OneLakeModelIds = [
    "aaaaaaaa-aaaa-aaaa-aaaa-0000000000a2",  # Azure Data Partner & Community
    "aaaaaaaa-aaaa-aaaa-aaaa-0000000000a1",  # Azure Data Insights
]

# Build the list of models to refresh
_is_group_refresh = not ItemId or ItemId == "OneLakeSecurityError-Group"
models_to_refresh = OneLakeModelIds if _is_group_refresh else [ItemId]

In [ ]:
import sempy.fabric as fabric

In [ ]:
import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from notebookutils import mssparkutils

workspace_id = fabric.resolve_workspace_id(WorkspaceName)

# Load .NET assembly for TOM
_loader = fabric.create_tom_server(readonly=True, workspace=workspace_id)
_loader.Dispose()
import Microsoft.AnalysisServices.Tabular as TOM

def send_xmla_refresh(dataset_name, refresh_type):
    """Send XMLA refresh command (synchronous). Raises on XMLA-level errors."""
    token = mssparkutils.credentials.getToken("https://analysis.windows.net/powerbi/api")
    conn_str = f"Provider=MSOLAP;Data Source=powerbi://api.powerbi.com/v1.0/myorg/{WorkspaceName};Password={token};"
    tmsl = json.dumps({"refresh": {"type": refresh_type, "objects": [{"database": dataset_name}]}})

    server = TOM.Server()
    server.Connect(conn_str)
    try:
        result = server.Execute(tmsl)
        for i in range(result.Count):
            for j in range(result[i].Messages.Count):
                msg = result[i].Messages[j]
                if hasattr(msg, "Description") and msg.Description:
                    raise RuntimeError(msg.Description)
    finally:
        server.Disconnect()
        server.Dispose()

def _step1_automatic_refresh(current_item_id):
    """Step 1 worker: resolve name and send automatic refresh for one model."""
    dataset_name = fabric.resolve_dataset_name(current_item_id, workspace_id)
    print(f"\n{'='*60}")
    print(f"Refreshing: {dataset_name} ({current_item_id})")
    print(f"{'='*60}")

    start_ms = int(time.time() * 1000)
    automatic_failed = False
    print(f"Step 1: sending automatic refresh for {dataset_name}...")
    try:
        send_xmla_refresh(dataset_name, "automatic")
        print(f"  Automatic refresh command completed for {dataset_name}.")
    except RuntimeError as e:
        print(f"  Automatic refresh error for {dataset_name}: {e}")
        automatic_failed = True

    return current_item_id, {"dataset_name": dataset_name, "automatic_failed": automatic_failed, "start_ms": start_ms}

refresh_results = {}
failed_models = []

with ThreadPoolExecutor(max_workers=len(models_to_refresh)) as executor:
    futures = {executor.submit(_step1_automatic_refresh, mid): mid for mid in models_to_refresh}
    for future in as_completed(futures):
        mid = futures[future]
        try:
            item_id, result = future.result()
            refresh_results[item_id] = result
        except Exception as e:
            print(f"\n*** ERROR refreshing model {mid}: {e} ***")
            failed_models.append(mid)

In [ ]:
# Step 2: Verify automatic refresh appeared in history, fall back to full if not
def _step2_verify_and_full(current_item_id):
    """Step 2 worker: verify automatic refresh, fall back to full if needed."""
    r = refresh_results[current_item_id]
    dataset_name = r["dataset_name"]
    automatic_failed = r["automatic_failed"]
    start_ms = r["start_ms"]

    print(f"\n{'='*60}")
    print(f"Verifying: {dataset_name} ({current_item_id})")
    print(f"{'='*60}")

    if automatic_failed:
        need_full = True
        print(f"Automatic refresh failed for {dataset_name}, falling back to full refresh.")
    else:
        refreshes = fabric.list_refresh_requests(dataset=current_item_id, workspace=workspace_id)
        need_full = True
        if refreshes is not None and len(refreshes) > 0:
            latest = refreshes.iloc[0]
            latest_start = int(latest["Start Time"].timestamp() * 1000) if hasattr(latest["Start Time"], "timestamp") else int(latest["Start Time"])
            if latest_start > start_ms:
                status = latest["Status"]
                request_id = latest["Request Id"]
                print(f"Automatic refresh verified for {dataset_name} (request: {request_id}, status: {status})")
                if status == "Completed":
                    need_full = False
                else:
                    print(f"Automatic refresh status was '{status}' for {dataset_name}, falling back to full refresh.")
        if need_full and not automatic_failed:
            print(f"Automatic refresh did not appear in history (no-op) for {dataset_name}, falling back to full refresh.")

    if need_full:
        print(f"Sending full refresh for {dataset_name}...")
        send_xmla_refresh(dataset_name, "full")
        print(f"Full refresh completed successfully for {dataset_name}")

candidates = [mid for mid in models_to_refresh if mid in refresh_results]

with ThreadPoolExecutor(max_workers=len(candidates)) as executor:
    futures = {executor.submit(_step2_verify_and_full, mid): mid for mid in candidates}
    for future in as_completed(futures):
        mid = futures[future]
        try:
            future.result()
        except Exception as e:
            dataset_name = refresh_results.get(mid, {}).get("dataset_name", "unknown")
            print(f"\n*** ERROR verifying/refreshing model {mid} ({dataset_name}): {e} ***")
            failed_models.append(mid)

print(f"\n{'='*60}")
print(f"All {len(models_to_refresh)} model(s) processed.")
if failed_models:
    print(f"WARNING: {len(failed_models)} model(s) had errors: {failed_models}")
print(f"{'='*60}")